# reentry_participation_counterfactual_v1

Purpose: test whether the shared SOXL poor-window failure mode is mainly a re-entry participation problem that can be improved by a narrow feature-gated replay overlay before spending another PPO training run.

Governance:
- Baseline: `current_runtime_paper` daily replay from `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`.
- Direct comparison: same replay with only the daily `position` column adjusted by `ReentryParticipationOverlayConfig`.
- Role: `diagnose`, not a promotion candidate unless it improves excess while keeping MaxDD within the target envelope across slices.
- Changed layer: `execution` replay overlay; no model weights or broker path changes.
- Target windows: full OOS replay, fixed walk-forward slices, and the aggregate bad clusters C001/C002/C003.
- Acceptance: prefer >= +30pp return/excess improvement versus baseline, no worse full MaxDD, and all key slice MaxDD <= 15%.

External references used for discipline, not copied logic:
- Microsoft Qlib paper and project for repeatable research harness framing: https://arxiv.org/abs/2009.11189 and https://github.com/microsoft/qlib
- Probability of Backtest Overfitting for keeping this as a small grid and avoiding retrospective promotion: https://scholarworks.wmich.edu/math_pubs/42/
- Deflated Sharpe / multiple testing caution for grid selection: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Behavior-cloning trading-agent literature as the next path if fixed overlays cannot express the behavior: https://arxiv.org/abs/2110.01325


In [1]:
from __future__ import annotations

import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.budget_counterfactual import compute_metrics, dashboard_daily_frame, slice_metrics
from src.soxl_quant.app.oos_diagnostics import load_qlib_feature_frame
from src.soxl_quant.app.participation_counterfactual import (
    ReentryParticipationOverlayConfig,
    apply_reentry_participation_overlay,
)

EXPERIMENT = "reentry_participation_counterfactual_v1"
OUTPUT_DIR = Path("runtime/execution/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
AGG_DIR = Path("runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate")

WINDOWS = {
    "wf_2020": ("2020-01-02", "2020-12-31"),
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C001_2020_rebound": ("2020-03-18", "2020-06-19"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}

daily = pd.read_csv(DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
price = daily.set_index("datetime")["price"].astype(float)
features = load_qlib_feature_frame(str(daily["datetime"].min().date()), str(daily["datetime"].max().date()))
poor_clusters = pd.read_csv(AGG_DIR / "poor_window_clusters.csv")
feature_clusters = pd.read_csv(AGG_DIR / "feature_delta_clusters.csv")

print({
    "daily_rows": len(daily),
    "daily_start": str(daily["datetime"].min().date()),
    "daily_end": str(daily["datetime"].max().date()),
    "features_shape": features.shape,
})
feature_clusters.head(8)

[61017:MainThread](2026-05-25 12:09:21,236) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61017:MainThread](2026-05-25 12:09:22,022) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61017:MainThread](2026-05-25 12:09:22,025) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[61017:MainThread](2026-05-25 12:09:24,711) INFO - qlib.timer - [log.py:127] - Time cost: 0.003s | Loading data Done


[61017:MainThread](2026-05-25 12:09:24,716) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[61017:MainThread](2026-05-25 12:09:24,717) INFO - qlib.timer - [log.py:127] - Time cost: 0.008s | Init data Done


{'daily_rows': 1605, 'daily_start': '2020-01-02', 'daily_end': '2026-05-21', 'features_shape': (1605, 59)}


,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score
0,vol_of_vol_21,5,2.909053,2.909053,1.0,-0.029445,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.212323
1,tracking_error_vol_21,4,3.153269,3.153269,1.0,0.008294,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074991
2,decay_63,5,-2.831926,2.831926,1.0,0.065458,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074129
3,realized_vol_63,5,2.528318,2.528318,1.0,-0.015127,"balanced_rule_residual_v19_reg,balanced_rule_r...",4.530138
4,tracking_error_21,4,-2.161666,2.161666,1.0,0.090702,"balanced_rule_residual_v19_reg,balanced_rule_r...",3.479067
5,realized_vol_21,5,1.623577,1.623577,1.0,-0.003108,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.909059
6,vol_coef_21,5,1.464646,1.464646,1.0,-0.003431,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.624293
7,vix_level,4,1.270719,1.270719,1.0,0.097231,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.045143


In [2]:
base_frame = daily[["datetime", "position"]].copy()
base_metrics = compute_metrics(base_frame, price)
base_slices = slice_metrics(base_frame, price, WINDOWS)

def flatten_metrics(prefix: str, metrics: dict) -> dict:
    return {
        f"{prefix}_return": metrics["return"],
        f"{prefix}_relative_return": metrics["relative_return"],
        f"{prefix}_max_dd": metrics["max_dd"],
        f"{prefix}_avg_pos": metrics["avg_pos"],
        f"{prefix}_trades": metrics["trades"],
    }

base_summary = {"window": "test_proxy", **flatten_metrics("base", base_metrics)}
for name, metrics in base_slices.items():
    base_summary[name] = flatten_metrics("base", metrics)

print(json.dumps(base_metrics, indent=2))
pd.DataFrame([
    {"window": "test_proxy", **base_metrics},
    *[{"window": name, **metrics} for name, metrics in base_slices.items()],
])[["window", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

{
  "rows": 1605,
  "start": "2020-01-02",
  "end": "2026-05-21",
  "return": 1542.8945209142537,
  "sharpe": 1.1088637871454499,
  "max_dd": 41.58720675168147,
  "avg_pos": 0.4622723738606798,
  "stress_avg_pos": 0.438732446228528,
  "stress_days": 697,
  "buy_and_hold_return": 816.5124997770641,
  "relative_return": 726.3820211371896,
  "trades": 199
}


,window,return,relative_return,max_dd,avg_pos,trades
0,test_proxy,1542.894521,726.382021,41.587207,0.462272,199
1,wf_2020,154.596623,94.821315,25.267996,0.536420,30
2,wf_2022,-21.567182,65.020890,31.096849,0.248277,26
3,wf_2023_2024,209.141657,17.368148,30.668184,0.497231,70
4,bad_C001_2020_rebound,76.658648,-101.802909,17.169856,0.609927,8
5,bad_C002_2025_rebound,76.882481,-124.606649,16.324924,0.572851,9
6,bad_C003_2026_reentry,17.799626,-217.582972,6.421473,0.161722,2


## Grid Design

The grid is intentionally small. It varies only the participation floor, the maximum base position eligible for a lift, and feature gates highlighted by the aggregate OOS diagnostic: `vol_of_vol_21`, `tracking_error_vol_21`, `vix_percentile_252`, plus momentum/relative-strength confirmation.

This keeps the experiment a counterfactual diagnostic rather than a hidden model-selection sweep.

In [3]:
feature_quantiles = features[["vol_of_vol_21", "tracking_error_vol_21", "vix_percentile_252", "mom_21", "relative_strength_21"]].quantile([0.40, 0.50, 0.60, 0.75]).T
feature_quantiles

,0.40,0.50,0.60,0.75
vol_of_vol_21,0.082583,0.096143,0.111792,0.138048
tracking_error_vol_21,0.052294,0.056584,0.060360,0.070772
vix_percentile_252,0.334524,0.484127,0.596032,0.761905
mom_21,0.026382,0.086317,0.145980,0.250790
relative_strength_21,0.130895,0.550878,0.964736,1.660773


In [4]:
def score_overlay(config: ReentryParticipationOverlayConfig) -> tuple[dict, pd.DataFrame, dict[str, dict]]:
    overlay = apply_reentry_participation_overlay(daily, features, config)
    overlay_metrics = compute_metrics(overlay[["datetime", "position"]], price)
    overlay_slices = slice_metrics(overlay[["datetime", "position"]], price, WINDOWS)
    row = {
        "min_target": config.min_target,
        "max_target": config.max_target,
        "max_base_position": config.max_base_position,
        "min_mom_21": config.min_mom_21,
        "min_relative_strength_21": config.min_relative_strength_21,
        "min_vol_of_vol_21": config.min_vol_of_vol_21,
        "min_tracking_error_vol_21": config.min_tracking_error_vol_21,
        "max_vix_percentile_252": config.max_vix_percentile_252,
        "max_equity_drawdown": config.max_equity_drawdown,
        "lifted_days": int(overlay["reentry_participation_lifted"].sum()),
        "ready_days": int(overlay["reentry_participation_ready"].sum()),
        **flatten_metrics("overlay", overlay_metrics),
        "delta_return": overlay_metrics["return"] - base_metrics["return"],
        "delta_relative_return": overlay_metrics["relative_return"] - base_metrics["relative_return"],
        "delta_max_dd": overlay_metrics["max_dd"] - base_metrics["max_dd"],
        "delta_avg_pos": overlay_metrics["avg_pos"] - base_metrics["avg_pos"],
    }
    max_slice_dd = 0.0
    slice_dd_pass_count = 0
    slice_delta_returns = []
    for name, metrics in overlay_slices.items():
        base = base_slices[name]
        row[f"{name}_return"] = metrics["return"]
        row[f"{name}_delta_return"] = metrics["return"] - base["return"]
        row[f"{name}_relative_return"] = metrics["relative_return"]
        row[f"{name}_max_dd"] = metrics["max_dd"]
        row[f"{name}_delta_max_dd"] = metrics["max_dd"] - base["max_dd"]
        row[f"{name}_avg_pos"] = metrics["avg_pos"]
        max_slice_dd = max(max_slice_dd, metrics["max_dd"])
        slice_dd_pass_count += int(metrics["max_dd"] <= 15.0)
        slice_delta_returns.append(metrics["return"] - base["return"])
    row["max_key_slice_dd"] = max_slice_dd
    row["slice_dd_pass_count"] = slice_dd_pass_count
    row["mean_slice_delta_return"] = float(np.mean(slice_delta_returns)) if slice_delta_returns else 0.0
    row["full_gate"] = bool(row["delta_return"] >= 30.0 and row["overlay_max_dd"] <= 15.0)
    row["non_regression_gate"] = bool(row["delta_return"] >= 30.0 and row["delta_max_dd"] <= 0.0)
    row["slice_gate"] = bool(slice_dd_pass_count == len(overlay_slices))
    row["diagnostic_score"] = (
        row["delta_return"]
        + 0.25 * row["mean_slice_delta_return"]
        - 10.0 * max(row["delta_max_dd"], 0.0)
        - 2.0 * max(row["max_key_slice_dd"] - 15.0, 0.0)
    )
    return row, overlay, overlay_slices

grid = []
configs = []
for min_target, max_base, min_mom, min_rs, min_vov, min_te, max_vix, max_eq_dd in product(
    [0.45, 0.55, 0.65],
    [0.25, 0.55],
    [-0.20, 0.00, 0.08],
    [-2.0, 0.0],
    [0.10, 0.12],
    [None, 0.08],
    [1.00, None],
    [0.08, 0.12],
):
    max_target = min(0.85, min_target + 0.10)
    cfg = ReentryParticipationOverlayConfig(
        enabled=True,
        min_target=min_target,
        max_target=max_target,
        max_base_position=max_base,
        min_mom_21=min_mom,
        min_relative_strength_21=min_rs,
        min_vol_of_vol_21=min_vov,
        min_tracking_error_vol_21=min_te,
        max_vix_percentile_252=max_vix,
        max_equity_drawdown=max_eq_dd,
    )
    row, _, _ = score_overlay(cfg)
    grid.append(row)
    configs.append(cfg)

print({"grid_configs": len(grid)})
results = pd.DataFrame(grid)
results["rank_gate"] = results["full_gate"].astype(int) * 100 + results["non_regression_gate"].astype(int) * 10 + results["slice_gate"].astype(int)
results = results.sort_values(["rank_gate", "diagnostic_score", "delta_return"], ascending=[False, False, False]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "grid_results.csv", index=False)
results.head(15)[[
    "min_target", "max_base_position", "min_mom_21", "min_relative_strength_21", "min_vol_of_vol_21",
    "min_tracking_error_vol_21", "max_vix_percentile_252", "max_equity_drawdown", "lifted_days",
    "overlay_return", "delta_return", "overlay_max_dd", "delta_max_dd", "max_key_slice_dd",
    "bad_C002_2025_rebound_delta_return", "bad_C003_2026_reentry_delta_return",
    "full_gate", "non_regression_gate", "slice_gate", "diagnostic_score"
]]

{'grid_configs': 576}


,min_target,max_base_position,min_mom_21,min_relative_strength_21,min_vol_of_vol_21,min_tracking_error_vol_21,max_vix_percentile_252,max_equity_drawdown,lifted_days,overlay_return,delta_return,overlay_max_dd,delta_max_dd,max_key_slice_dd,bad_C002_2025_rebound_delta_return,bad_C003_2026_reentry_delta_return,full_gate,non_regression_gate,slice_gate,diagnostic_score
0,0.65,0.25,-0.20,-2.0,0.10,NaN,1.0,0.08,50,2982.525857,1439.631336,39.536787,-2.050420,31.096849,0.0,90.583027,False,True,False,1412.843787
1,0.65,0.25,-0.20,-2.0,0.10,NaN,NaN,0.08,50,2982.525857,1439.631336,39.536787,-2.050420,31.096849,0.0,90.583027,False,True,False,1412.843787
2,0.65,0.25,-0.20,-2.0,0.12,NaN,1.0,0.08,46,2909.656949,1366.762428,39.798022,-1.789185,37.080395,0.0,78.385780,False,True,False,1326.182161
3,0.65,0.25,-0.20,-2.0,0.12,NaN,NaN,0.08,46,2909.656949,1366.762428,39.798022,-1.789185,37.080395,0.0,78.385780,False,True,False,1326.182161
4,0.65,0.25,0.08,-2.0,0.12,NaN,1.0,0.08,27,2842.783778,1299.889257,39.525307,-2.061899,30.668184,0.0,78.385780,False,True,False,1272.106885
5,0.65,0.25,0.08,-2.0,0.12,NaN,NaN,0.08,27,2842.783778,1299.889257,39.525307,-2.061899,30.668184,0.0,78.385780,False,True,False,1272.106885
6,0.65,0.25,0.08,0.0,0.12,NaN,1.0,0.08,27,2842.783778,1299.889257,39.525307,-2.061899,30.668184,0.0,78.385780,False,True,False,1272.106885
7,0.65,0.25,0.08,0.0,0.12,NaN,NaN,0.08,27,2842.783778,1299.889257,39.525307,-2.061899,30.668184,0.0,78.385780,False,True,False,1272.106885
8,0.55,0.25,0.08,-2.0,0.12,NaN,1.0,0.08,27,2596.866403,1053.971882,39.886797,-1.700410,30.668184,0.0,64.083452,False,True,False,1025.556655
9,0.55,0.25,0.08,-2.0,0.12,NaN,NaN,0.08,27,2596.866403,1053.971882,39.886797,-1.700410,30.668184,0.0,64.083452,False,True,False,1025.556655


In [5]:
best_row = results.iloc[0].to_dict()
best_cfg = ReentryParticipationOverlayConfig(
    enabled=True,
    min_target=float(best_row["min_target"]),
    max_target=float(best_row["max_target"]),
    max_base_position=float(best_row["max_base_position"]),
    min_mom_21=float(best_row["min_mom_21"]),
    min_relative_strength_21=float(best_row["min_relative_strength_21"]),
    min_vol_of_vol_21=float(best_row["min_vol_of_vol_21"]),
    min_tracking_error_vol_21=None if pd.isna(best_row["min_tracking_error_vol_21"]) else float(best_row["min_tracking_error_vol_21"]),
    max_vix_percentile_252=None if pd.isna(best_row["max_vix_percentile_252"]) else float(best_row["max_vix_percentile_252"]),
    max_equity_drawdown=float(best_row["max_equity_drawdown"]),
)
_, best_overlay, best_slices = score_overlay(best_cfg)
selected_daily = dashboard_daily_frame(best_overlay[["datetime", "position"]], price)
# Preserve audit columns that dashboard_daily_frame intentionally omits.
audit_cols = [
    "datetime", "base_position", "reentry_participation_feature_ready",
    "reentry_participation_equity_allowed", "reentry_participation_ready",
    "reentry_participation_lifted", "equity_nav", "equity_drawdown",
    "mom_21", "relative_strength_21", "vol_of_vol_21", "tracking_error_vol_21", "vix_percentile_252",
]
audit = best_overlay[[col for col in audit_cols if col in best_overlay.columns]].copy()
selected_daily = selected_daily.merge(audit, on="datetime", how="left")
selected_daily.to_csv(OUTPUT_DIR / f"{EXPERIMENT}_selected_soxl_daily.csv", index=False)

comparison_rows = []
comparison_rows.append({"window": "test_proxy", "variant": "baseline", **base_metrics})
comparison_rows.append({"window": "test_proxy", "variant": "selected_overlay", **compute_metrics(best_overlay[["datetime", "position"]], price)})
for name in WINDOWS:
    comparison_rows.append({"window": name, "variant": "baseline", **base_slices[name]})
    comparison_rows.append({"window": name, "variant": "selected_overlay", **best_slices[name]})
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "diagnostic_not_promoted",
    "reason": "best fixed participation lift improves return and slightly reduces full drawdown, but full/key-slice MaxDD remains above the <= 15% target",
    "baseline_daily_csv": str(DAILY_CSV),
    "selected_daily_csv": str(OUTPUT_DIR / f"{EXPERIMENT}_selected_soxl_daily.csv"),
    "grid_results_csv": str(OUTPUT_DIR / "grid_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_config": best_cfg.__dict__,
    "selected_metrics": {k: best_row[k] for k in sorted(best_row) if isinstance(best_row[k], (int, float, bool, str)) or pd.isna(best_row[k])},
    "next_path": "train or imitate a reentry participation policy with drawdown-aware labels; fixed post-hoc floor is too blunt",
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")

comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,variant,return,relative_return,max_dd,avg_pos,trades
0,test_proxy,baseline,1542.894521,726.382021,41.587207,0.462272,199
1,test_proxy,selected_overlay,2982.525857,2166.013357,39.536787,0.480549,204
2,wf_2020,baseline,154.596623,94.821315,25.267996,0.536420,30
3,wf_2020,selected_overlay,178.074994,118.299686,25.267996,0.564523,30
4,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
5,wf_2022,selected_overlay,-21.567182,65.020890,31.096849,0.248277,26
6,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70
7,wf_2023_2024,selected_overlay,224.827861,33.054351,30.668184,0.511869,70
8,bad_C001_2020_rebound,baseline,76.658648,-101.802909,17.169856,0.609927,8
9,bad_C001_2020_rebound,selected_overlay,76.658648,-101.802909,17.169856,0.609927,8


In [6]:
lifted = best_overlay[best_overlay["reentry_participation_lifted"]].copy()
cluster_lift = []
for name, (start, end) in WINDOWS.items():
    sub = lifted[(lifted["datetime"] >= start) & (lifted["datetime"] <= end)]
    cluster_lift.append({
        "window": name,
        "lifted_days": int(len(sub)),
        "base_avg_position_on_lifted_days": float(sub["base_position"].mean()) if len(sub) else 0.0,
        "overlay_avg_position_on_lifted_days": float(sub["position"].mean()) if len(sub) else 0.0,
        "avg_equity_drawdown_on_lifted_days": float(sub["equity_drawdown"].mean()) if len(sub) else 0.0,
    })
cluster_lift = pd.DataFrame(cluster_lift)
cluster_lift.to_csv(OUTPUT_DIR / "selected_lift_by_window.csv", index=False)
cluster_lift

,window,lifted_days,base_avg_position_on_lifted_days,overlay_avg_position_on_lifted_days,avg_equity_drawdown_on_lifted_days
0,wf_2020,14,0.142148,0.65,0.012386
1,wf_2022,0,0.000000,0.00,0.000000
2,wf_2023_2024,13,0.084727,0.65,0.041180
3,bad_C001_2020_rebound,0,0.000000,0.00,0.000000
4,bad_C002_2025_rebound,0,0.000000,0.00,0.000000
5,bad_C003_2026_reentry,22,0.000000,0.65,0.013064


In [7]:
base_full = base_metrics
selected_full = compute_metrics(best_overlay[["datetime", "position"]], price)
pass_rows = {
    "delta_return_pp": selected_full["return"] - base_full["return"],
    "delta_relative_return_pp": selected_full["relative_return"] - base_full["relative_return"],
    "baseline_max_dd_pct": base_full["max_dd"],
    "selected_max_dd_pct": selected_full["max_dd"],
    "delta_max_dd_pp": selected_full["max_dd"] - base_full["max_dd"],
    "selected_lifted_days": int(best_overlay["reentry_participation_lifted"].sum()),
    "selected_ready_days": int(best_overlay["reentry_participation_ready"].sum()),
    "best_bad_C002_delta_return_pp": best_row.get("bad_C002_2025_rebound_delta_return"),
    "best_bad_C003_delta_return_pp": best_row.get("bad_C003_2026_reentry_delta_return"),
    "full_gate": bool(best_row["full_gate"]),
    "non_regression_gate": bool(best_row["non_regression_gate"]),
    "slice_gate": bool(best_row["slice_gate"]),
}
conclusion = f"""## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

The best replay overlay lifted {pass_rows['selected_lifted_days']} days and improved full OOS return by {pass_rows['delta_return_pp']:.2f}pp, but full MaxDD moved from {pass_rows['baseline_max_dd_pct']:.2f}% to {pass_rows['selected_max_dd_pct']:.2f}% ({pass_rows['delta_max_dd_pp']:+.2f}pp). That fails the MaxDD <= 15% acceptance target. The full-window non-regression gate is {"passed" if pass_rows['non_regression_gate'] else "failed"}, so this is useful diagnostic evidence but not a production overlay.

Bad-window effect: C002 delta return {pass_rows['best_bad_C002_delta_return_pp']:.2f}pp; C003 delta return {pass_rows['best_bad_C003_delta_return_pp']:.2f}pp. The fixed floor can add participation where the aggregate diagnostic pointed, but it is too blunt to control drawdown across the full replay.

Keep/drop: keep `src.soxl_quant.app.participation_counterfactual` as a reusable diagnostic helper; do not promote this parameterized overlay to production execution. Next experiment should be behavior-cloning or label-conditioned re-entry participation with drawdown-aware targets, not another post-hoc floor threshold sweep.

Artifacts: `grid_results.csv`, `selected_comparison.csv`, `selected_lift_by_window.csv`, `selection.json`, and `{EXPERIMENT}_selected_soxl_daily.csv`.
"""
display(Markdown(conclusion))
print(json.dumps(pass_rows, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

The best replay overlay lifted 50 days and improved full OOS return by 1439.63pp, but full MaxDD moved from 41.59% to 39.54% (-2.05pp). That fails the MaxDD <= 15% acceptance target. The full-window non-regression gate is passed, so this is useful diagnostic evidence but not a production overlay.

Bad-window effect: C002 delta return 0.00pp; C003 delta return 90.58pp. The fixed floor can add participation where the aggregate diagnostic pointed, but it is too blunt to control drawdown across the full replay.

Keep/drop: keep `src.soxl_quant.app.participation_counterfactual` as a reusable diagnostic helper; do not promote this parameterized overlay to production execution. Next experiment should be behavior-cloning or label-conditioned re-entry participation with drawdown-aware targets, not another post-hoc floor threshold sweep.

Artifacts: `grid_results.csv`, `selected_comparison.csv`, `selected_lift_by_window.csv`, `selection.json`, and `reentry_participation_counterfactual_v1_selected_soxl_daily.csv`.


{
  "delta_return_pp": 1439.6313357104812,
  "delta_relative_return_pp": 1439.6313357104814,
  "baseline_max_dd_pct": 41.58720675168147,
  "selected_max_dd_pct": 39.53678675334604,
  "delta_max_dd_pp": -2.0504199983354283,
  "selected_lifted_days": 50,
  "selected_ready_days": 50,
  "best_bad_C002_delta_return_pp": 0.0,
  "best_bad_C003_delta_return_pp": 90.58302701843586,
  "full_gate": false,
  "non_regression_gate": true,
  "slice_gate": false
}
